# Google Merchandise Store: месяц из жизни магазина

Куда идёт посетитель после открытия листинга? Что происходит, когда он возвращается в корзину оформлять заказ? Разберём оба вопроса с [retentioneering](https://retentioneering.com/docs) на январской выгрузке за 2021 год из [обезличенного демонстрационного датасета Google](https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset). В выгрузке 493 416 событий, 109 078 сессий и 92 983 идентификатора пользователя. Google отмечает, что обезличивание может влиять на согласованность полей.

**Как читать исследование**

| Раздел | На какой вопрос отвечаем |
|---|---|
| [1. Данные](#gms-data) и [2. Воронка](#gms-funnel) | Какие шаги сессии проходят последовательно? |
| [3. Вход на страницы одежды](#gms-landings) | Почему столько сессий заканчивается на хабе одежды? Что можно проверить на этой странице? |
| [4. Маршруты от корзины](#gms-checkout) | Какая часть потерь при оформлении приходится на шаги до и после ввода адреса? |

Отправные точки — два наблюдения. Около 88% сессий с входом на хаб одежды заканчиваются после одного шага обработанного потока против 56% у подстраниц. После первой корзины в 54% сессий нет ни входа в аккаунт, ни шага адреса. В каждом случае посмотрим, какие траектории стоят за этими числами и какую продуктовую гипотезу они подсказывают.

<a id="gms-data"></a>

## 1. Подготовка и обзор данных

Скачайте подготовленную январскую выгрузку [`gms.csv.gz`](https://drive.google.com/file/d/19tAZNl6IROsqXaZlTLWX3WMSIUuDzh-J/view?usp=drive_link), положите рядом с ноутбуком и выполните ячейки по порядку. В примерах используются Python-пакет `retentioneering` и его виджеты для Jupyter.

Сохранённые таблицы получены с Retentioneering 5.3.0. Для соседних событий с одинаковой временной меткой существенен порядок, заданный при обработке.

Одна строка соответствует записанному событию. У посетителя может быть несколько сессий, а страница может встретиться в сессии несколько раз. При расчёте конверсии сессий каждый подходящий `session_id` входит в знаменатель один раз.

### 1.1 Загрузка исходных данных

In [1]:
import pandas as pd
import retentioneering as rete

print("retentioneering", rete.__version__)

df = pd.read_csv("gms.csv.gz", compression="gzip", sep=";")
df["page_location"] = "https://shop.googlemerchandisestore.com" + df["page_location"]
df.head()

retentioneering 5.3.0


,user_id,session_id,event,timestamp,session_number,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location
0,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:34.089387,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
1,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:39.094280,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
2,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
3,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
4,1.000436e+07,10004358.089772267_1,PLP: Bags,2021-01-08 04:10:12.372423,1,mobile,Russia,google,cpc,<Other>,Bags,https://shop.googlemerchandisestore.com/Google...


### 1.2 Загрузка данных в Eventstream

[Eventstream](https://retentioneering.com/docs/eventstream) — центральный класс библиотеки retentioneering.
По сути он является тонкой обёрткой `pandas.DataFrame`, которая знает, что значит каждая колонка. Объявим это через
[схему](https://retentioneering.com/docs/eventstream#schema):

- **`path_cols`** регулирует что считать одной траекторией. Обычно это колонки вроде `user_id` и `session_id`. В исходных данных есть и то, и другое, поэтому в `path_cols` добавляем обе эти колонки: `user_id` для поведения за всё время и `session_id` для одного визита. Первый становится значением по умолчанию; любой метод принимает `path_col="session_id"`, чтобы переключиться.
- **`segment_cols`** — категориальные метки событий, разбивающие ивентстрим на группы, поведение которых хочется сравнивать в ходе анализа. В исходном датасете это `country`,`utm_source`, `utm_medium`, `utm_campaign`, `product_type`.
- **`custom_cols`** — дополнительные данные, которые потребуются для анализа. В нашем случае это `page_location`, который очень пригодится в кейсе 1 (см. секцию 3.2).

Колонки `event` и `timestamp` уже названы дефолтными именами, так что упоминать в схеме их не нужно.

In [2]:
SCHEMA = {
    "path_cols": ["user_id", "session_id"],
    "segment_cols": [
        "device", "country", "utm_source", "utm_medium", "utm_campaign", "product_type",
    ],
    "custom_cols": ["page_location"],
}

stream0 = rete.Eventstream(df, schema=SCHEMA)
stream0.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,subindex,index
0,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2.0,1
1,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2.0,2
2,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2.0,1
3,1.000557e+06,1000557.2911835024_1,view_promotion,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2.0,2
4,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2.0,3


К исходным данным можно обратиться через аксессор `Eventstream.df`. Как мы видим, к оригинальным данным добавились три технические колонки: `event_type`, `index`, `subindex`.

Первый метод, который стоит вызвать в анализе – [`Eventstream.describe()`](https://retentioneering.com/docs/eventstream#describe). Он покажет базовую информацию о загруженном ивентстриме: размер, диапазон дат, частоты событий, распределение длин траекторий, уровни сегментов.

In [3]:
overview = stream0.describe(top_events=12)

print(overview["shape"])
print(overview["date_range"])
display(overview["path_stats"]["user_id"])
display(overview["event_frequency"])

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 47}
{'min': Timestamp('2021-01-01 00:00:08.136569'), 'max': Timestamp('2021-01-31 23:59:55.412363'), 'span': Timedelta('30 days 23:59:47.275794')}


,length,duration
count,92983.000000,9.298300e+04
mean,5.306518,3.612019e+04
std,10.437298,1.891713e+05
min,1.000000,0.000000e+00
25%,2.000000,4.894917e+00
50%,2.000000,5.045870e+00
75%,5.000000,6.255278e+01
90%,10.000000,3.211639e+03
99%,49.000000,1.129364e+06
max,593.000000,2.614472e+06


,event,count,share
0,main,116795,0.236707
1,PLP: Apparel,65391,0.132527
2,view_promotion,52209,0.105811
3,PDP: Apparel,36969,0.074925
4,PLP: Shop By Brand,25923,0.052538
5,store,23871,0.048379
6,PLP: Lifestyle,21323,0.043215
7,basket,18680,0.037859
8,add_to_cart,15179,0.030763
9,select_item,9949,0.020164


Для дальнейшего анализа интересны две детали. Среди 47 имён событий встречаются варианты `PLP` (листинг товаров) и `PDP` (карточка товара) для разных категорий. А у половины траекторий, сгруппированных по посетителю, не более двух событий. Короткие траектории пригодятся, когда мы будем разбирать окончания сессий.

<a id="gms-funnel"></a>

## 2. Обзор пользовательского поведения

### 2.1 Наивный первый шаг

Давайте попробуем сразу построить один из наиболее важных виджетов retentioneering – [transition graph](https://retentioneering.com/docs/widgets/transition-graph) – и убедимся, что его прямолинейное применение не даст желаемого результата.

In [4]:
stream0.transition_graph(path_col="session_id")

TransitionGraphWidget — выполните ячейку, чтобы открыть виджет.

Даже с [автоматическим фильтром рёбер](https://retentioneering.com/docs/widgets/transition-graph#edgefilter) граф с 47 именами событий трудно читать.

### 2.2 Объединяем листинги и карточки товаров

И `PLP: Bags`, и `PLP: Drinkware` означают посещение товарного листинга. Для первого обзора сохраним тип страницы и пока отвлечёмся от категории товара. [`rename_events`](https://retentioneering.com/docs/data-processors/rename-events) сводит 14 имён PLP и 11 имён PDP к двум событиям. В обзоре останется 24 имени события. В разделе 3 вернёмся к деталям из исходного потока.

In [5]:
plp_pdp = {e: e.split(":")[0] for e in stream0.df["event"].unique() if e.startswith(("PLP:", "PDP:"))}

list(plp_pdp.items())[:4]

[('PLP: Clearance', 'PLP'),
 ('PLP: Lifestyle', 'PLP'),
 ('PLP: Apparel', 'PLP'),
 ('PDP: Apparel', 'PDP')]

In [6]:
stream = stream0.rename_events(plp_pdp)
stream.describe()["shape"]

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 24}

### 2.3 Схлопываем повторы внутри сессии

Серия `shipping_details` может соответствовать нескольким правкам адреса, а серия `add_to_cart` — добавлению нескольких товаров. Сейчас нам важно, что происходит *после* каждой серии. [`collapse_events(loops=True)`](https://retentioneering.com/docs/data-processors/collapse-events) представляет идущие подряд одинаковые имена событий как один шаг.

Передаём `path_col="session_id"`, чтобы серии заканчивались на границе сессии. Дальше «следующий шаг» означает следующее событие обработанной сессии. На таком уровне детализации даже листинги разных категорий, открытые подряд, могут схлопнуться в один `PLP`.

In [7]:
stream = stream.collapse_events(loops=True, path_col="session_id")

print(f"{len(stream0.df):,} событий -> {len(stream.df):,} после схлопывания повторов")

493,416 событий -> 325,330 после схлопывания повторов


Таким образом, мы облегчили наш ивентстрим примерно на треть.

> Фильтр, который мы намеренно *не* применили. Обычно хочется выбросить совсем короткие
> траектории с аргументом "сессии из 1-2 событий это шум, их всё равно не проанализировать". В
> нашем случае это не стоит делать. В кейсе 1 мы покажем, что в коротких сессиях и находится один из инсайтов (см. секцию 3).

### 2.4 Тот же граф, теперь читабельный

Теперь граф переходов выглядит так:

In [8]:
stream.transition_graph(path_col="session_id")

TransitionGraphWidget — выполните ячейку, чтобы открыть виджет.

Расположение вершин на графе автоматическое, и обычно оно требует ручной доработки. Чтобы сохранить граф полностью, используйте [`state_file`](https://retentioneering.com/docs/widgets#saving-widget-state). Тогда состояние всех его элементов, в том числе расположение вручную расставленных вершин, фильтров и т.д. переживёт перезапуск ноутбука:

```python
stream.transition_graph(state_file="gms_transition_graph.json", path_col="session_id")
```

Теперь на графе можно разглядеть некоторую структуру CJM. Но чтобы отобразить её явно, воспользуемся отображением пути на графе c помощью механизма [views](https://retentioneering.com/docs/widgets/transition-graph#views):

In [9]:
happy_path = ["PLP", "PDP", "add_to_cart", "basket", "shipping_details", "payment_details", "purchase"]

stream.transition_graph(
    views=[{"name": "Happy path", "focus": {"type": "path", "nodes": happy_path}}],
    view="Happy path",
    path_col="session_id"
)

TransitionGraphWidget — выполните ячейку, чтобы открыть виджет.

Подсвеченный маршрут соединяет соседние события на графе. Сессия может пройти те же этапы с промежуточным поиском, входом в аккаунт или другими событиями. Посчитаем такие сессии в упорядоченной воронке.

### 2.5 Исследование happy path

Давайте теперь оценим конверсию между шагами happy path с точки зрения [воронки](https://retentioneering.com/docs/widgets/funnel). Шаги воронки упорядоченные и закрытые: сессия попадает на шаг N, только если прошла все предыдущие шаги в нужном порядке.

Разобьём для удобства этот путь на две части: до начала чекаута и после.

In [10]:
stream.funnel(steps=["PLP", "PDP", "add_to_cart"], path_col="session_id", height=500)

FunnelWidget — выполните ячейку, чтобы открыть виджет.

Из 109 078 сессий в 54 947 (50,4%) есть листинг, в 8 503 (7,8%) после листинга есть карточка товара, а в 1 426 (1,3%) за ними следует `add_to_cart`. Последние два процента — доли **всех сессий**, в которых наблюдалась указанная последовательность. Если взять только сессии, достигшие предыдущего шага воронки, получим 8 503 / 54 947 = 15,5% и 1 426 / 8 503 = 16,8%.

Теперь начнём новую воронку от корзины. В неё сессия может попасть любым маршрутом.

In [11]:
stream.funnel(
    steps=["basket", "shipping_details", "payment_details", "purchase"],
    path_col="session_id",
    height=460,
)

FunnelWidget — выполните ячейку, чтобы открыть виджет.

Из 6 743 сессий с корзиной 2 078 затем доходят до `shipping_details` (30,8%). Из них 1 524 переходят к `payment_details` (73,3%), а из этих сессий 1 092 доходят до `purchase` (71,7%). На первом шаге теряется больше всего сессий и по количеству, и по доле. К нему вернёмся в разделе 4.

<a id="gms-landings"></a>

## 3. Почему теряются пользователи на этапе выбора товара?

### 3.1 Где заканчиваются сессии

Чтобы посмотреть, какие шаги предшествовали `path_end`, удобно воспользоваться [Step matrix](https://retentioneering.com/docs/widgets/step-matrix) или [Step Sankey](https://retentioneering.com/docs/widgets/step-sankey) с аргументом `path_pattern="path_end"`, который выровняет траектории по последнему событию.

In [12]:
stream.step_matrix(path_pattern="path_end", path_col="session_id", height=360)

StepMatrixWidget — выполните ячейку, чтобы открыть виджет.

В 39% сессий последнее записанное событие — `PLP`, а в 21% — `main`. Частые маркеры `path_start` в нескольких шагах от конца также показывают, что много сессий короткие. На какие листинги приходятся выходы?

Похожую информацию можно было получить с помощью transition graph. Удобнее всего узнать о том, какие события предшествовали `path_end` можно с помощью режима [**Ego view**](https://retentioneering.com/docs/widgets/transition-graph#ego-view) с фокусом на вершину `path_end`. В этом режиме граф разворачивает окрестность одного события в Sankey диаграмму и показывает все входящие и исходящие рёбра с весами `proba_in` для входящих рёбер и `proba_out` для исходящих. В случае фокуса на `path_end` покажутся только входящие рёбра, которые и будут соответствовать событиям, завершающим сессию.

In [13]:
stream.transition_graph(
    path_col="session_id",
    views=[{
        "name": "Where sessions end",
        "egoNode": "path_end",                             # открывает модалку Ego view при применении
        "focus": {"type": "node", "event": "path_end"},    # и оставляет узел сфокусированным под ней
    }],
    view="Where sessions end",
    height=560,
)

TransitionGraphWidget — выполните ячейку, чтобы открыть виджет.

Итак, мы видим тот же топ событий, завершающих сессию, который мы видели в step matrix: `PLP` (39%), `main` (21%), `PDP` (16%).

### 3.2 Анализ отдельных страниц

Давайте копнём чуть глубже в сторону `PLP` и попробуем разобраться, почему эта страница часто становится последней в сессии пользователя. Выше мы намеренно схлопывали все страницы товарных категорий в `PLP`, чтобы получить более понятную высокоуровневую картину. Теперь нам нужно наоборот углубиться в конкретные товарные категории, чтобы проверить, нет ли среди них таких страниц, которые влияют на отток больше других. Здесь нам пригодится колонка `page_location`, которую мы обработаем с помощью дата процессора [`urls_to_events`](https://retentioneering.com/docs/data-processors/urls-to-events). Он собирает имена событий из URL, позволяя схлопывать ветки URL-дерева на разных уровнях.

Обратите внимание на порядок: сначала мы разделяем по URL, потом схлопываем повторяющиеся события, как делали это в прошлой секции.

In [14]:
all_pages = (
    stream0
    .urls_to_events("page_location", nodes=[], keep_full_paths=True)
    .collapse_events(loops=True, path_col="session_id")
)

print(f"{all_pages.df['event'].nunique()} различных событий с URL")

931 различных событий с URL


In [15]:
all_pages.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,index,subindex
0,1.000300e+06,1000299.7413851356_1,main://,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,collapsed,1,2.0
1,1.000557e+06,1000557.2911835024_1,main://,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,1,2.0
2,1.000557e+06,1000557.2911835024_1,view_promotion://,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,2.0
3,1.000557e+06,1000557.2911835024_1,main://,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,3,2.0
4,1.000557e+06,1000557.2911835024_1,PLP: Clearance:/google+redesign/clearance,2021-01-07 12:15:56.604844,mobile,India,<Other>,<Other>,<Other>,Clearance,https://shop.googlemerchandisestore.com/Google...,raw,4,2.0


Для каждого события с URL зададим два вопроса. Какая доля *всех окончаний сессий* приходится на это событие? Какая доля *исходящих из него переходов* ведёт сразу в `path_end`? В [данных графа переходов](https://retentioneering.com/docs/widgets/transition-graph#headless-mode) первое соответствует `proba_in`, второе — `proba_out`. Добавим число записанных вхождений события, чтобы видеть объём за процентами.

In [16]:
share_of_exits = all_pages.transition_graph_data(
    edge_weight="proba_in", path_col="session_id"
)["path_end"]
exit_rate = all_pages.transition_graph_data(
    edge_weight="proba_out", path_col="session_id"
)["path_end"]

exits = pd.DataFrame({
    "share of all exits": share_of_exits,
    "event occurrences": pd.Series(all_pages.get_event_counts()),
    "exit rate": exit_rate,
}).sort_values("share of all exits", ascending=False)

with pd.option_context("display.max_colwidth", 70):
    display(exits.head(10).round(3))

,share of all exits,event occurrences,exit rate
main://,0.213,72431.0,0.320
PLP: Apparel:/google+redesign/apparel,0.154,22986.0,0.733
view_promotion://,0.095,44429.0,0.232
PDP: Apparel:/google+redesign/apparel/google+dino+game+tee,0.065,7722.0,0.922
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube,0.055,9458.0,0.630
store:/store.html,0.049,16469.0,0.326
PLP: Apparel:/google+redesign/apparel/mens,0.021,10719.0,0.212
PLP: Lifestyle:/google+redesign/lifestyle/drinkware,0.020,6193.0,0.359
basket:/basket.html,0.019,12328.0,0.171
PLP: Clearance:/google+redesign/clearance,0.018,8897.0,0.222


На хаб одежды приходится 15,4% всех окончаний сессий. Среди исходящих переходов из хаба 73,3% ведут в `path_end`. У мужского листинга соответствующие доли — 2,1% и 21,2%. Может ли высокий показатель выходов с хаба объясняться частыми входами на эту страницу? Посмотрим отдельно на сессии, которые начинаются с хаба или заходят на него позднее.

### 3.3 Exit rate, разделённый по сценариям

Чтобы сравнить хаб с его же подстраницами, они сначала должны стать двумя разными событиями. `aggregate_children` в `urls_to_events` оставляет URL верхнего уровня `/google+redesign/apparel` отдельным событием и сворачивает все его подстраницы в `sub-page`. Дальше `rename_events` возвращает URL к читаемым именам.

In [17]:
pages = stream0.urls_to_events(
    "page_location",
    nodes=[{"path": "/google+redesign/apparel", "aggregate_children": True, "name": "sub-page"}],
).collapse_events(loops=True, path_col="session_id")

pages.df["event"].value_counts().head(8)

event
main://                                                      72431
view_promotion://                                            44429
PDP: Apparel:/google+redesign/apparel/sub-page               23449
PLP: Apparel:/google+redesign/apparel                        22986
PLP: Apparel:/google+redesign/apparel/sub-page               19336
store:/store.html                                            16469
basket:/basket.html                                          12328
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube     9458
Name: count, dtype: int64

In [18]:
SPLIT = {
    "PLP: Apparel:/google+redesign/apparel": "Apparel hub",
    "PLP: Apparel:/google+redesign/apparel/sub-page": "Apparel sub-page",
    "PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube": "YouTube hub",
}

# оставляем три разделения выше; у всех остальных отрезаем URL и возвращаем обычное имя
pages = pages.rename_events({e: SPLIT.get(e, e.split(":")[0]) for e in pages.df["event"].unique()})

pages.df["event"].value_counts().head(8)

event
main                72515
PLP                 61900
view_promotion      50624
PDP                 41750
Apparel hub         22986
Apparel sub-page    19336
store               16487
basket              12439
Name: count, dtype: int64

Теперь с помощью [`Eventstream.get_conversion_rate()`](https://retentioneering.com/docs/eventstream#get_conversion_rate) можно посчитать конверсии из некоторого события (якоря) `start_anchor` в наступление `end_anchor="path_end"` на следующем шаге. В качестве `start_anchor` нас будут интересовать следующие [паттерны](https://retentioneering.com/docs/path-patterns):

- `path_start->PAGE` – страница была первым событием сессии;
- `.->PAGE` – перед страницей было некоторое настоящее событие (`.` намеренно не матчит `path_start`).

Аргумент `within=1` потребует, чтобы `end_anchor="path_end"` произошёл ровно на следующем шаге после якоря, задаваемого `start_anchor`. Список из нескольких `start_anchor` посчитает конверсии в `end_anchor` для каждого из них.

Конверсии понимаются именно в смысле траекторий: доля траекторий, в которых встретился `start_anchor`, и потом после которого наступил `end_anchor`.

In [19]:
listing_pages = ["Apparel hub", "Apparel sub-page", "YouTube hub"]
page_anchors = [
    f"{lead}->{page}"
    for page in listing_pages
    for lead in ("path_start", ".")
]
next_step_exits = pages.get_conversion_rate(
    start_anchor=page_anchors, end_anchor="path_end", within=1, path_col="session_id",
)
display(next_step_exits[[
    "start_anchor", "paths_with_start", "converted", "conversion_rate"
]].round(3))

raw_session_sizes = df["session_id"].value_counts()
single_event_ids = raw_session_sizes.index[raw_session_sizes.eq(1)]
single_event_pages = pages.df.loc[pages.df["session_id"].isin(single_event_ids), "event"]
single_event_pages.value_counts().reindex(listing_pages, fill_value=0).rename("sessions with one raw event")

,start_anchor,paths_with_start,converted,conversion_rate
0,path_start->Apparel hub,17392,15292,0.879
1,.->Apparel hub,4340,1351,0.311
2,path_start->Apparel sub-page,2298,1277,0.556
3,.->Apparel sub-page,9835,2331,0.237
4,path_start->YouTube hub,7289,5200,0.713
5,.->YouTube hub,1834,668,0.364


event
Apparel hub         1271
Apparel sub-page     135
YouTube hub          189
Name: sessions with one raw event, dtype: int64

Из 17 392 сессий с началом на хабе 15 292 (87,9%) переходят в `path_end` на следующем шаге обработанного потока. Для входов на подстраницы эта доля равна 1 277 / 2 298 (55,6%). Мы сгруппировали URL-события и схлопнули повторы, поэтому несколько просмотров хаба подряд могут выглядеть как один шаг. Ровно одно исходное событие содержат 1 271 сессия с входом на хаб и 135 сессий с входом на подстраницу. Для оценки времени на странице нужна отдельная метрика.

Если посещению хаба предшествует другое событие, 1 351 / 4 340 сессий (31,1%) переходят в `path_end` на следующем шаге после первого такого совпадения. Одна сессия может начаться на хабе и вернуться к нему позже, поэтому эти два знаменателя могут пересекаться. Что увидим, если проследить траекторию дальше ближайшего шага?

### 3.4 Продвижение по воронке в зависимости от лэндинга

На всякий случай проверим дополнительно, сохраняется ли конверсия из `path_start->Apparel hub` низкой не только с точки зрения быстрого завершения сессии, но и с точки зрения продвижения по воронке. Теперь в качестве `end_anchor` выберем не `path_end`, а поочерёдно `PDP`, `basket`, `purchase`, а также уберём условие `within=1`.

In [20]:
apparel_pages = ["Apparel hub", "Apparel sub-page"]
apparel_anchors = [
    f"{lead}->{page}"
    for lead in ("path_start", ".")
    for page in apparel_pages
]
landing_outcomes = pages.get_conversion_rate(
    start_anchor=apparel_anchors,
    end_anchor=["PDP", "basket", "purchase"],
    path_col="session_id",
)

landing_outcomes[[
    "start_anchor", "end_anchor", "paths_with_start", "converted", "conversion_rate", "lift"
]].round(4)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,lift
0,path_start->Apparel hub,PDP,17392,495,0.0285,0.1254
1,path_start->Apparel hub,basket,17392,230,0.0132,0.2139
2,path_start->Apparel hub,purchase,17392,22,0.0013,0.1262
3,path_start->Apparel sub-page,PDP,2298,446,0.1941,0.8548
4,path_start->Apparel sub-page,basket,2298,227,0.0988,1.5979
5,path_start->Apparel sub-page,purchase,2298,59,0.0257,2.5622
6,.->Apparel hub,PDP,4340,1281,0.2952,1.3000
7,.->Apparel hub,basket,4340,697,0.1606,2.5979
8,.->Apparel hub,purchase,4340,172,0.0396,3.9551
9,.->Apparel sub-page,PDP,9835,3743,0.3806,1.6763


После входа на хаб покупка записана в 22 из 17 392 сессий (0,13%), а после входа на подстраницу — в 59 из 2 298 (2,57%). Разрыв примерно двадцатикратный. Карточки и корзины также встречаются чаще после входа на подстраницу. Каждый исход измеряем отдельно от начального якоря.

Чтобы получить `lift`, наблюдаемую конверсию делим на долю всех сессий с тем же исходом. При сравнении двух групп по одному исходу этот общий знаменатель сокращается. Поэтому отношение значений `lift` повторяет отношение конверсий. Возможно ли, что у страниц разная входящая аудитория?

### 3.5 Лэндинг как сегмент

Признак "Началась ли эта сессия на `Apparel hub`" можно сделать не разовым фильтром, а зафиксировать как [сегмент](https://retentioneering.com/docs/segments). Дата процессор [`Eventstream.add_segment()`](https://retentioneering.com/docs/data-processors/add-segment) с аргументом `sql` позволяет гибко назначать уровни сегмента для каждой строки. Мы немного расширим этот признак и зададим сегмент с тремя уровнями в зависимости от первого события в сессии: `Apparel hub`, `Apparel sub-page` или `elsewhere` – они запишутся в колонку `landing`.

In [21]:
pages = pages.add_segment(
    "landing",
    sql="""
        SELECT CASE FIRST_VALUE(event) OVER (PARTITION BY session_id ORDER BY "index")
            WHEN 'Apparel hub' THEN 'Apparel hub'
            WHEN 'Apparel sub-page' THEN 'Apparel sub-page'
            ELSE 'elsewhere'
        END
        FROM eventstream
    """,
    path_col="session_id"
)

[Segment overview](https://retentioneering.com/docs/widgets/segment-overview) позволяет сравнить длину и длительность сессий и частоты событий между группами входа. Для строк со *входом* на хаб или подстраницу из раздела 3.4 метрика `has_event` воспроизводит доли покупок, корзин и карточек. Событие входа задаёт группу, и каждое из этих событий отличается от первого, поэтому встретиться в этой сессии оно может только позже. Для строк с посещением страницы внутри сессии сохраняем отсчёт от соответствующего якоря.

Посмотрим и на строки UTM. По каждому источнику, каналу или кампании видна доля сессий, содержащих эту метку хотя бы один раз. Так можно изучить, как размечен входящий трафик.

In [22]:
pages.segment_overview(
    "landing",
    path_col="session_id",
    metrics=[
        {"metric": "length"},
        {"metric": "duration"},
        {"metric": "has_event", "metric_args": {"event": "PDP"}},
        {"metric": "has_event", "metric_args": {"event": "basket"}},
        {"metric": "has_event", "metric_args": {"event": "purchase"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_source"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_medium"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_campaign"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "device"}},
        {"metric": "in_segment", "metric_args": {"segment_name": "country", "segment_level": ["United States", "India", "Canada", "United Kingdom"]}},
    ],
)

SegmentOverviewWidget — выполните ячейку, чтобы открыть виджет.

Каждая строка UTM показывает долю сессий, в которых хотя бы один раз встретилась соответствующая метка. В одной сессии может быть несколько меток. Чтобы посмотреть на пару «источник и канал» в момент входа, возьмём первое событие каждой сессии, начавшейся на хабе.

In [23]:
hub_sessions = (
    pages.df.loc[pages.df["landing"].eq("Apparel hub")]
    .sort_values(["session_id", "index", "subindex"])
    .drop_duplicates("session_id")
)
source_counts = hub_sessions.groupby(
    ["utm_source", "utm_medium"], observed=True, dropna=False
).size().sort_values(ascending=False)

source_counts.head(5).to_frame("sessions").assign(
    share_of_hub_landings=lambda table: table["sessions"] / len(hub_sessions)
).round(4)

sessions  share_of_hub_landings
utm_source                      utm_medium                                 
google                          organic         5968                 0.3431
(direct)                        (none)          4127                 0.2373
<Other>                         <Other>         2986                 0.1717
                                referral        1679                 0.0965
shop.googlemerchandisestore.com referral         929                 0.0534

Самая частая записанная пара — `google / organic`, 5 968 из 17 392 входов на хаб (34,3%). Посетители с одинаковыми источником и каналом могли прийти по разным запросам или искать разные товары. Посмотрим, какие первые шаги делают посетители магазина.

### 3.6 Сравниваем первые шаги

В [step matrix в режиме diff](https://retentioneering.com/docs/widgets#diff-mode) каждая ячейка показывает долю события на данном шаге в первой группе за вычетом доли во второй. Первой группой возьмём входы на хаб, второй — входы на подстраницу.

In [24]:
landing_comparison = ("landing", "Apparel hub", "Apparel sub-page")
pages.step_matrix(
    diff=landing_comparison, path_col="session_id", step_window=5, height=380
)

StepMatrixWidget — выполните ячейку, чтобы открыть виджет.

На шаге после входа `path_end` встречается у хаба примерно на 32 п.п. чаще. Чтобы увидеть следующие действия посетителей, которые продолжили сессию, выровняем их траектории по второму шагу обработанного потока. [`truncate_paths`](https://retentioneering.com/docs/data-processors/truncate-paths) оставит сессии, в которых такое событие есть.

In [25]:
continued = pages.truncate_paths(
    start_anchor={"pattern": "path_start->.->.", "at": 2},
    end_anchor="path_end",
    path_col="session_id",
)
continued.step_matrix(
    diff=landing_comparison, path_col="session_id", step_window=5, height=380
)

StepMatrixWidget — выполните ячейку, чтобы открыть виджет.

Среди продолжившихся сессий на первом выровненном шаге после входа на хаб чаще встречаются `search` и `view_search_results` — на 15,6 и 16,1 п.п. соответственно. Подстраницы одежды также чаще встречаются на этом шаге. Возможное объяснение — посетитель ищет через поиск категорию или товар, который ожидал увидеть прямо на витрине. Содержание запросов помогло бы проверить эту версию.

Представить тогдашние страницы позволяют [архивная версия хаба](https://web.archive.org/web/20210123165920/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel) и [мужского листинга](https://web.archive.org/web/20210304130937/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Mens).

### 3.7 Что проверить на хабе одежды?

Высокая доля сессий, заканчивающихся на листинге, привела нас к одной странице. Хаб выделяется как точка входа. Его входные сессии часто состоят из одного шага обработанного потока, а среди продолжившихся сессий следующий шаг часто связан с поиском. На такую картину могут влиять состав аудитории, поисковые запросы и устройство самого хаба.

Продуктовая команда могла бы сразу показать на входе подкатегории и популярные товары. При первом подходящем визите случайно распределим посетителей между текущим и изменённым хабом, сохраним вариант при повторных визитах и сравним посещения карточек и покупки на распределённого посетителя за фиксированное окно. Так можно оценить эффект изменения страницы для тех, кто приходит на хаб.

<a id="gms-checkout"></a>

## 4. От корзины до покупки

Что происходит между открытием корзины и покупкой? Сначала проследим за сессией от первой корзины, затем разберём отдельные возвраты в корзину и в конце соберём все маршруты вместе. В этом разделе считаем уникальные сессии.

### 4.1 Где расходятся маршруты?

Начнём отсчёт от первой `basket` в каждой сессии. Для каждого целевого события посчитаем сессии, где оно встречается позднее в том же визите. В таблицах `get_conversion_rate` ниже знаменатель каждой строки указан в столбце `paths_with_start`.

In [26]:
def percent_view(frame, *rate_columns):
    view = frame.copy()
    for name in rate_columns:
        view[name] = view[name].map("{:.2%}".format, na_action="ignore")
    return view

basket_baseline = stream.get_conversion_rate(
    start_anchor="basket", end_anchor=["shipping_details", "purchase"], path_col="session_id"
).set_index("end_anchor")
rate_columns = ["paths_with_start", "converted", "conversion_rate"]
percent_view(basket_baseline[rate_columns], "conversion_rate")

,paths_with_start,converted,conversion_rate
end_anchor,,,
shipping_details,6743,2078,30.82%
purchase,6743,1092,16.19%


Из 6 743 сессий с корзиной 2 078 (30,82%) затем доходят до шага адреса, а в 1 092 (16,19%) есть покупка. В корзину можно зайти, чтобы отложить товар, уточнить сумму или начать оформление. Посмотрим, какие следующие шаги видны в логе.

#### 4.1.1 Что следует за корзиной?

Выровняем step matrix по первой `basket` и посмотрим на столбец справа от неё.

In [27]:
stream.step_matrix(path_pattern="basket", path_col="session_id")

StepMatrixWidget — выполните ячейку, чтобы открыть виджет.

Примерно в 24% сессий за первой корзиной следует `sign_in`, ещё примерно в 24% — `path_end`. На листинг приходится 21%, на карточку товара — 10%, на `shipping_details` — 5%.

Расширим вопрос. В течение сессии в корзину можно вернуться. Если где-либо встречается соседняя пара `basket → sign_in` или `basket → shipping_details`, как часто за ней следует покупка? В каждой строке отсчёт начинается в конце первого совпадения с её парой.

In [28]:
sign_in_pair = "basket->sign_in"
shipping_pair = "basket->shipping_details"
pair_rates = stream.get_conversion_rate(
    start_anchor=[sign_in_pair, shipping_pair], end_anchor="purchase", path_col="session_id"
).set_index("start_anchor")
percent_view(pair_rates[rate_columns], "conversion_rate")

,paths_with_start,converted,conversion_rate
start_anchor,,,
basket->sign_in,2258,681,30.16%
basket->shipping_details,1589,833,52.42%


Конверсии равны 681 / 2 258 (30,16%) и 833 / 1 589 (52,42%). Во второй группе отсчёт уже начинается с шага адреса. Есть и ещё одна деталь. В одной сессии можно пройти первым маршрутом, вернуться в корзину и пройти вторым. Измерим пересечение, сохранив для каждой сессии по одному признаку на паттерн.

In [29]:
def session_flags(patterns):
    """Return a Boolean column per named pattern, indexed by session ID."""
    metrics = [
        {"metric": "matches_pattern", "metric_args": {"pattern": pattern}}
        for pattern in patterns.values()
    ]
    values = stream.get_metrics(metrics, path_col="session_id")
    columns = [f"matches_pattern_{pattern}" for pattern in patterns.values()]
    flags = values[columns].set_axis(list(patterns), axis="columns")
    assert flags.index.is_unique and flags.notna().all().all()
    return flags.astype(bool)

pairs = session_flags({"sign_in": sign_in_pair, "shipping": shipping_pair})
assert pairs.sum().tolist() == pair_rates.loc[
    [sign_in_pair, shipping_pair], "paths_with_start"
].tolist()
pd.Series({
    "sign_in only": (pairs["sign_in"] & ~pairs["shipping"]).sum(),
    "both": pairs.all(axis=1).sum(),
    "shipping only": (pairs["shipping"] & ~pairs["sign_in"]).sum(),
}, name="sessions")

sign_in only     1340
both              918
shipping only     671
Name: sessions, dtype: int64

#### 4.1.2 На каком шаге теряются сессии после входа?

В обе группы попали 918 сессий. Теперь проследим только за 2 258 сессиями с `basket → sign_in`. Сколько из них затем дошли до адреса и сколько после этого купили?

In [30]:
def checkout_stages(anchor):
    return session_flags({
        "eligible": anchor,
        "shipping": f"{anchor}->.*->shipping_details",
        "purchase": f"{anchor}->.*->purchase",
        "ordered_purchase": f"{anchor}->.*->shipping_details->.*->purchase",
    })

after_sign_in = checkout_stages(sign_in_pair)
after_sign_in.sum().rename("sessions")

eligible            2258
shipping            1397
purchase             681
ordered_purchase     681
Name: sessions, dtype: int64

In [31]:
assert after_sign_in["eligible"].equals(pairs["sign_in"])
assert after_sign_in["purchase"].sum() == pair_rates.at[sign_in_pair, "converted"]
bypass = after_sign_in["purchase"] & ~after_sign_in["ordered_purchase"]
assert not bypass.any(), f"{bypass.sum()} purchase sessions lack a shipping → purchase sequence"

После входа адрес встречается в 1 397 / 2 258 сессиях (61,87%). Из них 681 / 1 397 (48,75%) затем дошли до покупки. Во всех сессиях с покупкой после этой пары проверка по ID нашла последовательность «адрес → покупка». Отсюда получается разложение

$$\frac{681}{2258} = \frac{1397}{2258}\times\frac{681}{1397}.$$

В 861 сессии за парой не последовал адрес. Ещё в 716 адрес есть, но после него нет покупки. Это два разных места для дальнейшего исследования.

Посмотрим на следующий шаг в этих 861 сессии. Паттерн «любая пара со входом, после которой нет адреса» выбрал бы 867 сессий. В шести из них адрес уже был после более ранней пары со входом. Отберём 861 сессию по ID и выровняем матрицу по концу первой пары.

In [32]:
no_shipping = after_sign_in["eligible"] & ~after_sign_in["shipping"]
no_shipping_ids = after_sign_in.index[no_shipping].tolist()
stream.filter_events(keep={"session_id": no_shipping_ids}).step_matrix(
    anchor={"pattern": sign_in_pair, "at": -1}, path_col="session_id"
)

StepMatrixWidget — выполните ячейку, чтобы открыть виджет.

После входа примерно в 31% из 861 сессии следующий шаг — конец сессии, в 19% — `registration`, а в 18% — `basket`. Вернёмся к полному потоку, где переход к адресу по-прежнему возможен, и сравним исходы для трёх следующих шагов.

In [33]:
registration = f"{sign_in_pair}->registration"
next_steps = stream.get_conversion_rate(
    start_anchor=[registration, f"{sign_in_pair}->basket", f"{sign_in_pair}->PLP"],
    end_anchor=["shipping_details", "purchase"],
    path_col="session_id",
).set_index(["start_anchor", "end_anchor"])
percent_view(next_steps[rate_columns], "conversion_rate")

paths_with_start  converted  \
start_anchor                  end_anchor                                      
basket->sign_in->registration shipping_details               795        625   
                              purchase                       795        214   
basket->sign_in->basket       shipping_details               222         59   
                              purchase                       222         27   
basket->sign_in->PLP          shipping_details               102         14   
                              purchase                       102          4   

                                               conversion_rate  
start_anchor                  end_anchor                        
basket->sign_in->registration shipping_details          78.62%  
                              purchase                  26.92%  
basket->sign_in->basket       shipping_details          26.58%  
                              purchase                  12.16%  
basket->sign_in->PLP          shipping_details          13.73%  
                              purchase                   3.92%

В 795 сессиях встретилась последовательность `basket → sign_in → registration`. После неё до адреса дошли 625 сессий, а до покупки — 214. Что происходит между событием регистрации и возвратом к заказу?

#### 4.1.3 Как сессии возвращаются в оформление после регистрации?

In [34]:
stream.step_matrix(path_pattern=registration, path_col="session_id", step_window=7)

StepMatrixWidget — выполните ячейку, чтобы открыть виджет.

Матрица подсказывает возможный маршрут через витрину. Мы видим `store` и `view_promotion` с последующим повторным `basket`. Посчитаем возврат двумя способами. Широкий паттерн допускает **ноль и более** событий витрины в промежутке, поэтому прямой возврат тоже подходит. Строгий паттерн требует хотя бы одного `store` в промежутке. В каждом паттерне сессия учитывается один раз независимо от количества попыток.

In [35]:
storefront_gap = "[store|view_promotion]*"
basket_return = f"{registration}->{storefront_gap}->basket"
store_return = f"{registration}->{storefront_gap}->store->{storefront_gap}->basket"
routes = {
    "registration": checkout_stages(registration),
    "basket return": checkout_stages(basket_return),
    "basket return via store": checkout_stages(store_return),
}

assert (routes["basket return via store"]["eligible"] <= routes["basket return"]["eligible"]).all()
assert (routes["basket return"]["eligible"] <= routes["registration"]["eligible"]).all()
for route, flags in routes.items():
    assert (flags["ordered_purchase"] <= (flags["shipping"] & flags["purchase"])).all(), route
    assert flags["purchase"].equals(flags["ordered_purchase"]), f"Check purchase order for {route}"

route_counts = pd.DataFrame({name: flags.sum() for name, flags in routes.items()}).T
route_counts["purchase / eligible"] = route_counts["purchase"] / route_counts["eligible"]
route_counts["ordered_purchase / shipping"] = route_counts["ordered_purchase"] / route_counts["shipping"]
percent_view(route_counts, "purchase / eligible", "ordered_purchase / shipping")

,eligible,shipping,purchase,ordered_purchase,purchase / eligible,ordered_purchase / shipping
registration,795,625,214,214,26.92%,34.24%
basket return,549,499,180,180,32.79%,36.07%
basket return via store,514,478,169,169,32.88%,35.36%


В широкой группе после возврата 499 из 549 сессий доходят до адреса, а 180 из 549 (32,79%) — до покупки. Для всех 180 сессий с покупкой подтвердился порядок «возврат → адрес → покупка». Если начать отсчёт с адреса, конверсия составляет **180 / 499 = 36,07%**.

В 37 сессиях первое широкое совпадение — прямой возврат из регистрации в корзину. В двух из них позднее произошёл и возврат через `store`. С учётом всех попыток в строгую группу входят 514 сессий. Из них 478 доходят до адреса, а 169 затем покупают (35,36% от 478). Лог показывает маршрут. Для проверки автоматического редиректа нужны событие успешной регистрации и цель навигации. Теперь посчитаем возвраты в корзину, допустив в промежутке любые события.

In [36]:
all_returns = stream.get_conversion_rate(
    start_anchor=registration, end_anchor="basket", path_col="session_id"
).set_index("end_anchor")
percent_view(all_returns[rate_columns], "conversion_rate")

,paths_with_start,converted,conversion_rate
end_anchor,,,
basket,795,710,89.31%


Любым маршрутом в корзину вернулись 710 из 795 сессий (89,31%). В остальных 85 повторной корзины в этой сессии нет. Если отбирать для сравнения только вернувшихся, все 85 выпадут из знаменателя. Поэтому в продуктовом тесте в конце раздела начнём отсчёт до навигации.

#### 4.1.4 Каков вклад каждой ветки?

Для декомпозиции вернёмся к **первой корзине** каждой сессии. Ищем ближайший вход или адрес. Если сначала встретился вход, ищем ближайшую регистрацию или адрес. Эти развилки задают шесть строк ниже. Между развилками могут встречаться другие события.

Отнесём каждую сессию с корзиной ровно к одной строке, затем в каждой строке посчитаем покупки **после первой корзины**. Единая точка отсчёта важна для веток, паттерн которых заканчивается на `path_end`. При поиске покупки за этой границей ноль получился бы по определению.

In [37]:
first_basket = "path_start->[^basket]*->basket"
before_sign_or_shipping = "[^sign_in|shipping_details]*"
before_registration_or_shipping = "[^registration|shipping_details]*"
before_shipping = "[^shipping_details]*"

from_basket = f"{first_basket}->{before_sign_or_shipping}"
from_sign_in = f"{from_basket}->sign_in->{before_registration_or_shipping}"
from_registration = f"{from_sign_in}->registration->{before_shipping}"
branches = {
    "no sign-in or shipping": f"{from_basket}->path_end",
    "shipping before sign-in": f"{from_basket}->shipping_details",
    "sign-in → shipping": f"{from_sign_in}->shipping_details",
    "sign-in → registration → shipping": f"{from_registration}->shipping_details",
    "sign-in → registration, no shipping": f"{from_registration}->path_end",
    "sign-in, no registration or shipping": f"{from_sign_in}->path_end",
}

membership = session_flags(branches)
basket_ids = stream.df.loc[stream.df["event"].eq("basket"), "session_id"].unique()
expected_memberships = membership.index.isin(basket_ids).astype(int)
bad = membership.sum(axis=1).ne(expected_memberships)
assert not bad.any(), f"Incorrect branch membership for {membership.index[bad][:5].tolist()}"

`share` — доля ветки среди сессий с корзиной. `contribution_pp` — число сессий с покупкой в этой ветке на 100 сессий с корзиной. Эти вклады в процентных пунктах суммируются в базовую конверсию. `magnitude_pp` показывает отклонение вклада от того, каким он был бы при средней конверсии во всех ветках. Сумма отклонений равна нулю.

In [38]:
from math import isclose

outcomes = session_flags({
    "purchase": "basket->.*->purchase",
    "shipping": "basket->.*->shipping_details",
    "no_shipping": f"{first_basket}->{before_shipping}->path_end",
})
n = len(basket_ids)
p = int(outcomes["purchase"].sum())
assert n == basket_baseline.at["purchase", "paths_with_start"]
assert p == basket_baseline.at["purchase", "converted"]
shipping_branches = membership[["shipping before sign-in", "sign-in → shipping", "sign-in → registration → shipping"]]
assert shipping_branches.any(axis=1).equals(outcomes["shipping"])
remaining = membership.drop(columns=shipping_branches.columns)
assert remaining.any(axis=1).equals(outcomes["no_shipping"])

table = pd.DataFrame({
    "sessions": membership.sum(),
    "purchases": membership.loc[outcomes["purchase"]].sum(),
})
table["purchase_rate"] = table["purchases"] / table["sessions"]
table["share"] = table["sessions"] / n
table["contribution_pp"] = 100 * table["purchases"] / n
table["magnitude_pp"] = table["contribution_pp"] - table["share"] * (100 * p / n)

assert table["sessions"].sum() == n and table["purchases"].sum() == p
assert isclose(table["share"].sum(), 1.0)
assert isclose(table["contribution_pp"].sum(), 100 * p / n)
assert isclose(table["magnitude_pp"].sum(), 0.0, abs_tol=1e-10)
table.loc["total"] = [n, p, p / n, 1.0, 100 * p / n, 0.0]
table = table.astype({"sessions": int, "purchases": int})
view = percent_view(table, "purchase_rate", "share").round({"contribution_pp": 2, "magnitude_pp": 2})
view.set_axis([
    "1. Без входа и адреса",
    "2. Адрес до входа",
    "3. Вход → адрес",
    "4. Вход → регистрация → адрес",
    "5. Вход → регистрация, без адреса",
    "6. Вход без регистрации и адреса",
    "Итого",
])

,sessions,purchases,purchase_rate,share,contribution_pp,magnitude_pp
1. Без входа и адреса,3642,0,0.00%,54.01%,0.00,-8.75
2. Адрес до входа,630,377,59.84%,9.34%,5.59,4.08
3. Вход → адрес,807,492,60.97%,11.97%,7.30,5.36
4. Вход → регистрация → адрес,641,223,34.79%,9.51%,3.31,1.77
"5. Вход → регистрация, без адреса",216,0,0.00%,3.20%,0.00,-0.52
6. Вход без регистрации и адреса,807,0,0.00%,11.97%,0.00,-1.94
Итого,6743,1092,16.19%,100.00%,16.19,0.00


В самой крупной ветке 3 642 / 6 743 сессий (54,0%) после первой корзины не содержат ни входа, ни адреса. Ещё в 1 023 (15,2%) есть вход и ни одного события адреса после первой корзины. Если сессия сначала дошла до адреса, а позднее в ней появился вход, она здесь относится к ветке с адресом. Во всех трёх этих ветках после первой корзины нет покупок.

В ветках с адресом конверсия в покупку составляет 377 / 630 (59,84%), когда адрес встречается до входа, 492 / 807 (60,97%) после входа без промежуточной регистрации и 223 / 641 (34,79%) после входа и регистрации. Эти доли описывают сессии, дошедшие до адреса. Они подсказывают, где проверить доступ к оформлению и возврат к нему. Эффект изменения каждого сценария можно измерить в продуктовом тесте.

### 4.2 Что происходит в сессиях, которые не дошли до адреса?

Вернёмся к 4 665 сессиям без `shipping_details` после первой корзины. Это ветки 1, 5 и 6 из таблицы выше. Привязка к **первой** корзине существенна. Если посетитель дошёл до адреса, снова открыл корзину и ушёл, в этой сессии наш целевой шаг уже состоялся.

Обрежем каждую траекторию по первой корзине и применим [кластерный анализ](https://retentioneering.com/docs/widgets/cluster-analysis) к счётчикам оставшихся событий. По счётчикам видно, какие события обычно встречаются вместе. За порядком событий можно вернуться к отдельным траекториям.

In [39]:
post_basket_without_shipping = f"{first_basket}->{before_shipping}->path_end"
unfinished_checkout = stream.filter_paths(
    {
        "metric": "matches_pattern",
        "metric_args": {"pattern": post_basket_without_shipping},
        "op": "=",
        "value": True,
    },
    path_col="session_id",
).truncate_paths(start_anchor="basket", end_anchor="path_end", path_col="session_id")

expected_sessions = outcomes.index[outcomes["no_shipping"]]
assert set(unfinished_checkout.df["session_id"].unique()) == set(expected_sessions)

unfinished_checkout.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    overview_metrics=[{"metric": "length"}, {"metric": "event_count_bulk"}],
    method="kmeans",
    method_args={"n_clusters": "2-8"},
    nmf_components="2-8",
    select={"n_clusters": 3, "nmf_components": 3},
    path_col="session_id",
)

ClusterAnalysisWidget — выполните ячейку, чтобы открыть виджет.

В виджете можно посмотреть разные варианты разбиения. Для выбранных трёх кластеров откроем обзор и сравним размеры групп, длины путей и частые события.

| Сессии | Доля из 4 665 | Средняя длина пути от корзины | Что видно в событиях |
|---:|---:|---:|---|
| 4 271 | 91,6% | 3,24 | Короткие пути, иногда с повторным посещением корзины, листингом или входом |
| 314 | 6,7% | 23,01 | Повторные листинги, просмотры товаров и визиты в корзину |
| 80 | 1,7% | 8,96 | Выше средние счётчики страниц с информацией о доставке, ответами на вопросы и условиями возврата |

Например, в крупнейшем кластере на сессию приходится в среднем 0,22 события `sign_in`. Если нужна доля сессий хотя бы с одним входом, её показывает метрика `has_event`.

Небольшая группа с посещениями страниц про доставку и возврат подсказывает вопрос для следующего разбора. На каком шаге открывают эти страницы и что происходит сразу после? Для ответа можно просмотреть отдельные траектории и сопоставить их с журналом ошибок.

### 4.3 Какие продуктовые изменения можно проверить?

Из 6 743 сессий с корзиной в 1 023 после первой корзины встречается `sign_in`, а `shipping_details` отсутствует на всём отрезке от первой корзины до конца сессии. Другое наблюдение — 549 сессий, в которых после паттерна регистрации снова открыли корзину по изученному маршруту. Каждое наблюдение указывает на решение, которое можно изменить в продукте.

| Гипотеза | Когда распределять посетителей по вариантам | Контроль и тест |
|---|---|---|
| Возможность оформить заказ гостем увеличивает долю покупателей. | При первом запросе обязательного входа во время оформления с активной корзиной. | Текущий запрос входа или тот же экран с возможностью оформить заказ гостем. |
| Продолжение оформления после успешной регистрации увеличивает долю покупателей. | В момент подтверждённой успешной регистрации из оформления с активной корзиной, до перехода на следующую страницу. | Текущая страница назначения или продолжение оформления с той же корзиной. |

Для второго теста сначала запишем `registration_success`, источник перехода, запланированную и фактическую страницы назначения и факт восстановления той же корзины. Эти события позволят установить, как часто действительно происходит автоматический редирект.

В каждом тесте случайно распределим подходящих посетителей по вариантам в указанный момент и сохраним назначение при повторных визитах. Основная метрика — доля посетителей хотя бы с одной покупкой в фиксированном окне после назначения варианта, например за 24 часа. В знаменатель входят все распределённые посетители, включая тех, кто закрыл страницу или не вернулся в корзину. Оценим разницу долей в процентных пунктах с доверительным интервалом. Достижение шага адреса и восстановление корзины помогут понять, где изменился путь. Вместе с покупками будем отслеживать ошибки оформления и отмены заказов.

## 5. Заключение

Мы начали с перегруженного графа переходов и проследили два маршрута. Примерно в 88% сессий, начавшихся с хаба одежды, остаётся один шаг после объединения событий страниц и схлопывания последовательных повторов. Сравнение с входами на подстраницы подсказывает команде магазина конкретный вопрос о том, какие товары и категории стоит показать посетителю при входе.

При оформлении заказа повторные визиты в корзину позволяют одной сессии попасть сразу в несколько паттернов перехода. Когда мы привязали анализ к первой корзине, шесть веток охватили все сессии с корзиной и все 1 092 сессии с последующей покупкой. Такое разбиение показывает этапы, на которых теряются сессии. Эксперименты с гостевым оформлением и возвратом к заказу после регистрации позволят измерить, как каждое изменение влияет на долю покупателей.